In [1]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "streamlit"],
    capture_output=True, text=True
)
print("streamlit:", "✅ 已安装" if result.returncode == 0 else "❌ 缺失")

# 顺便确认 onnxruntime
result2 = subprocess.run(
    [sys.executable, "-m", "pip", "show", "onnxruntime"],
    capture_output=True, text=True
)
print("onnxruntime:", "✅ 已安装" if result2.returncode == 0 else "❌ 缺失")

streamlit: ✅ 已安装
onnxruntime: ✅ 已安装


In [2]:
import torch
import torch.nn as nn
import torch.onnx
import onnxruntime as ort
import numpy as np
import os

# 复用定义
class ListNet(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.scorer(x).squeeze(1)

data_dir = os.path.expanduser("~/ml-projects/stock-ranker/data")

ckpt = torch.load(f"{data_dir}/listnet_v2.pth", map_location="cpu")
model = ListNet(input_dim=8)
model.load_state_dict(ckpt["model_state"])
model.eval()

# 导出 ONNX
dummy_input = torch.randn(20, 8)  # 20只股票，8个特征
onnx_path = f"{data_dir}/listnet_v2.onnx"

torch.onnx.export(
    model, dummy_input, onnx_path,
    input_names=["features"],
    output_names=["scores"],
    dynamic_axes={"features": {0: "n_stocks"}, "scores": {0: "n_stocks"}},
    opset_version=14
)
print(f"已导出：{onnx_path}")

# 验证：PyTorch vs ONNX 输出对比
sess = ort.InferenceSession(onnx_path)
test_input = torch.randn(20, 8)

with torch.no_grad():
    pt_out = model(test_input).numpy()

ort_out = sess.run(["scores"], {"features": test_input.numpy()})[0]

print(f"最大误差：{np.abs(pt_out - ort_out).max():.8f}")

# 速度对比
import time
N = 1000
t0 = time.time()
for _ in range(N):
    with torch.no_grad(): model(test_input)
pt_ms = (time.time() - t0) / N * 1000

t0 = time.time()
for _ in range(N):
    sess.run(["scores"], {"features": test_input.numpy()})
ort_ms = (time.time() - t0) / N * 1000

print(f"PyTorch:  {pt_ms:.3f} ms")
print(f"ONNX:     {ort_ms:.3f} ms")
print(f"加速比:   {pt_ms/ort_ms:.2f}x")

/var/folders/lv/mnjxv0d53_xczbffr49m5z_40000gn/T/ipykernel_83751/1011554401.py:31: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0414 22:27:33.480000 83751 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ListNet([...]` with `torch.export.export(..., strict=False)`...


/opt/anaconda3/envs/mlenv/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).
Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/opt/anaconda3/envs/mlenv/lib/python3.11/site

[torch.onnx] Obtain model graph for `ListNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
已导出：/Users/tongwenchao/ml-projects/stock-ranker/data/listnet_v2.onnx
最大误差：0.00000012
PyTorch:  0.027 ms
ONNX:     0.005 ms
加速比:   5.49x


In [3]:
app_path = os.path.expanduser("~/ml-projects/stock-ranker/src/app.py")

code = '''
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import torch, torch.nn as nn
import onnxruntime as ort
import pickle, json, os, glob
from datetime import datetime

# ── 常量 ──────────────────────────────────────────────
DATA_DIR   = os.path.expanduser("~/ml-projects/stock-ranker/data")
OUTPUT_DIR = os.path.expanduser("~/ml-projects/stock-ranker/output")
FEATURE_COLS = ['mom_5d','mom_10d','mom_20d','vol_10d','vol_20d',
                'rsi_dist','high_20d_ratio','vol_ratio']
DEFAULT_TICKERS = [
    "AAPL","MSFT","GOOGL","AMZN","NVDA",
    "META","TSLA","AMD","INTC","CRM",
    "JPM","BAC","GS","MS","WFC",
    "JNJ","UNH","PFE","MRK","ABBV"
]

# ── 加载资源（缓存）──────────────────────────────────
@st.cache_resource
def load_resources():
    sess = ort.InferenceSession(f"{DATA_DIR}/listnet_v2.onnx")
    with open(f"{DATA_DIR}/scaler_v2.pkl", "rb") as f:
        scaler = pickle.load(f)
    return sess, scaler

# ── 特征工程 ─────────────────────────────────────────
def make_features(close, volume):
    f = pd.DataFrame(index=close.index)
    f["mom_5d"]         = close.pct_change(5)
    f["mom_10d"]        = close.pct_change(10)
    f["mom_20d"]        = close.pct_change(20)
    f["vol_10d"]        = close.pct_change().rolling(10).std()
    f["vol_20d"]        = close.pct_change().rolling(20).std()
    f["rsi_dist"]       = (close - close.rolling(20).mean()) / close.rolling(20).std()
    f["high_20d_ratio"] = close / close.rolling(20).max()
    f["vol_ratio"]      = volume / volume.rolling(10).mean()
    return f

def get_watch_list(tickers, sess, scaler, top_k=5):
    raw    = yf.download(tickers, period="2mo", interval="1d",
                         auto_adjust=True, progress=False)
    close  = raw["Close"]
    volume = raw["Volume"]

    rows = []
    for t in tickers:
        feat = make_features(close[t], volume[t]).dropna()
        if len(feat) == 0: continue
        latest = feat.iloc[[-1]]
        latest["ticker"] = t
        rows.append(latest)

    df = pd.concat(rows)
    latest_date = df.index.max()
    feats_scaled = scaler.transform(df[FEATURE_COLS].values)
    scores = sess.run(["scores"], {"features": feats_scaled.astype(np.float32)})[0]
    df["score"] = scores
    result = (df[["ticker","score"]]
              .sort_values("score", ascending=False)
              .head(top_k)
              .reset_index(drop=True))
    result.index += 1
    return result, latest_date

# ── UI ───────────────────────────────────────────────
st.set_page_config(page_title="Stock Ranker", page_icon="📈", layout="wide")
st.title("📈 Stock Watch List")
st.caption("Learning-to-Rank · ListNet · yfinance")

sess, scaler = load_resources()

with st.sidebar:
    st.header("设置")
    ticker_input = st.text_area(
        "股票池（每行一个）",
        value="\\n".join(DEFAULT_TICKERS), height=300
    )
    top_k  = st.slider("Watch list 数量", 3, 10, 5)
    run_btn = st.button("🔄 立即运行", use_container_width=True)

# ── 主区域 ───────────────────────────────────────────
col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("今日 Watch List")
    if run_btn:
        tickers = [t.strip().upper() for t in ticker_input.split("\\n") if t.strip()]
        with st.spinner("拉取数据 & 推理中..."):
            watch_list, date = get_watch_list(tickers, sess, scaler, top_k)
        st.success(f"数据日期：{date.date()}")
        st.dataframe(watch_list, use_container_width=True)

        # 保存 JSON
        out = {
            "date": str(date.date()),
            "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "watch_list": watch_list.rename(
                columns={"score":"model_score"}).to_dict(orient="records")
        }
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        with open(f"{OUTPUT_DIR}/{date.date()}.json", "w") as f:
            json.dump(out, f, indent=2)
    else:
        st.info("点击左侧「立即运行」生成 watch list")

with col2:
    st.subheader("历史记录")
    files = sorted(glob.glob(f"{OUTPUT_DIR}/*.json"), reverse=True)
    if files:
        for fp in files[:7]:
            with open(fp) as f:
                rec = json.load(f)
            with st.expander(f"📅 {rec['date']}  （生成于 {rec['generated_at']}）"):
                st.dataframe(pd.DataFrame(rec["watch_list"]), use_container_width=True)
    else:
        st.info("暂无历史记录")
'''

with open(app_path, "w") as f:
    f.write(code)
print("已保存 src/app.py")
print("\n启动命令：")
print("cd ~/ml-projects/stock-ranker/src && streamlit run app.py")

已保存 src/app.py

启动命令：
cd ~/ml-projects/stock-ranker/src && streamlit run app.py
